<a href="https://colab.research.google.com/github/Prabhjot-Singh1/Deep-Learning-Lab/blob/main/END_SEM_PRACTICAL/DLL_practical_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

In [2]:
BATCH_SIZE = 64
EPOCHS     = 10
LR         = 0.001
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

Using device: cuda


In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_set = torchvision.datasets.FashionMNIST(root='./data', train=True,
                                               download=True, transform=transform)
test_set  = torchvision.datasets.FashionMNIST(root='./data', train=False,
                                               download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False)

CLASSES = ['T-shirt','Trouser','Pullover','Dress','Coat',
           'Sandal','Shirt','Sneaker','Bag','Ankle Boot']

print("Fashion MNIST loaded!")

100%|██████████| 26.4M/26.4M [00:02<00:00, 10.5MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 163kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.18MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 26.4MB/s]


Fashion MNIST loaded!


In [4]:
# ── Base Model (no regularization) ──────────────────────────
class BaseModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 28→14
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 14→7
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),                  # 7→7
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU()                  # 7→7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 256), nn.ReLU(),
            nn.Linear(256, 10)
        )
    def forward(self, x):
        return self.classifier(self.features(x))


# ── Model with Dropout ───────────────────────────────────────
class DropoutModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU()
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),                          # randomly zeros 50% neurons
            nn.Linear(128 * 7 * 7, 256), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 10)
        )
    def forward(self, x):
        return self.classifier(self.features(x))


# ── Model with L2 + Dropout (Best Model) ────────────────────
class L2DropoutModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU()
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(128 * 7 * 7, 256), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 10)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

# NOTE:
# L1 and L2 regularization are NOT nn.Module layers in PyTorch.
# They are added manually in the training loop as a penalty to the loss.
# L2 is also available via weight_decay in the optimizer (shown below).

print("All models defined!")

All models defined!


In [5]:
def train(model, optimizer, epoch, l1=False, l2_lambda=0.0):
    model.train()
    correct, total, total_loss = 0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = nn.CrossEntropyLoss()(outputs, labels)

        # ── L1 Penalty (sum of |weights|) ───────────────────
        if l1:
            l1_penalty = sum(p.abs().sum() for p in model.parameters())
            loss = loss + 1e-4 * l1_penalty

        # ── L2 Penalty (sum of weights²) ────────────────────
        # Note: when using weight_decay in Adam, this is automatic.
        # Manual version shown here for understanding:
        if l2_lambda > 0:
            l2_penalty = sum(p.pow(2).sum() for p in model.parameters())
            loss = loss + l2_lambda * l2_penalty

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)

    return 100. * correct / total


def evaluate(model):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total   += labels.size(0)

    return 100. * correct / total


def run_experiment(name, model, optimizer, l1=False, l2_lambda=0.0):
    print(f"\n{'='*45}")
    print(f"  {name}")
    print(f"{'='*45}")
    model.to(DEVICE)

    for epoch in range(1, EPOCHS + 1):
        train_acc = train(model, optimizer, epoch, l1=l1, l2_lambda=l2_lambda)
        test_acc  = evaluate(model)
        print(f"Epoch {epoch:02d} | Train: {train_acc:.2f}%  Test: {test_acc:.2f}%")

    return train_acc, test_acc

In [6]:
results = {}

# a. Base Model
model = BaseModel()
optimizer = optim.Adam(model.parameters(), lr=LR)
results['Base Model'] = run_experiment("a. Base Model", model, optimizer)

# b. L1 Regularization (penalty added manually in train loop)
model = BaseModel()
optimizer = optim.Adam(model.parameters(), lr=LR)
results['L1 Reg'] = run_experiment("b. L1 Regularization", model, optimizer, l1=True)

# c. L2 Regularization (weight_decay in Adam = L2 penalty)
model = BaseModel()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
results['L2 Reg'] = run_experiment("c. L2 Regularization", model, optimizer)

# d. Dropout only
model = DropoutModel()
optimizer = optim.Adam(model.parameters(), lr=LR)
results['Dropout'] = run_experiment("d. Dropout", model, optimizer)

# e. L2 + Dropout
model = L2DropoutModel()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
results['L2 + Dropout'] = run_experiment("e. L2 + Dropout", model, optimizer)


  a. Base Model
Epoch 01 | Train: 84.26%  Test: 88.41%
Epoch 02 | Train: 90.09%  Test: 89.72%
Epoch 03 | Train: 91.99%  Test: 91.05%
Epoch 04 | Train: 93.22%  Test: 90.92%
Epoch 05 | Train: 94.02%  Test: 91.77%
Epoch 06 | Train: 94.94%  Test: 91.87%
Epoch 07 | Train: 95.87%  Test: 91.36%
Epoch 08 | Train: 96.58%  Test: 91.98%
Epoch 09 | Train: 97.25%  Test: 92.16%
Epoch 10 | Train: 97.72%  Test: 91.99%

  b. L1 Regularization
Epoch 01 | Train: 81.94%  Test: 86.35%
Epoch 02 | Train: 87.62%  Test: 88.02%
Epoch 03 | Train: 88.95%  Test: 88.64%
Epoch 04 | Train: 89.59%  Test: 88.66%
Epoch 05 | Train: 90.07%  Test: 88.91%
Epoch 06 | Train: 90.47%  Test: 89.53%
Epoch 07 | Train: 90.86%  Test: 89.57%
Epoch 08 | Train: 91.18%  Test: 89.87%
Epoch 09 | Train: 91.45%  Test: 90.15%
Epoch 10 | Train: 91.67%  Test: 89.77%

  c. L2 Regularization
Epoch 01 | Train: 83.96%  Test: 88.56%
Epoch 02 | Train: 89.93%  Test: 89.91%
Epoch 03 | Train: 91.49%  Test: 90.55%
Epoch 04 | Train: 92.63%  Test: 91.66%

In [7]:
print("\n")
print(f"{'='*50}")
print(f"{'Model':<20} {'Train Acc':>12} {'Test Acc':>12}")
print(f"{'='*50}")

for name, (train_acc, test_acc) in results.items():
    print(f"{name:<20} {train_acc:>11.2f}% {test_acc:>11.2f}%")

print(f"{'='*50}")



Model                   Train Acc     Test Acc
Base Model                 97.72%       91.99%
L1 Reg                     91.67%       89.77%
L2 Reg                     96.01%       91.86%
Dropout                    94.60%       92.48%
L2 + Dropout               93.90%       92.26%
